# CIS3120 Library Database
**CIS 3120: Programming for Analytics - Spring 2026**

---

## Housekeeping

In [ ]:
import sqlite3
import csv
import os
import urllib.request

# ── GitHub raw URLs ─────────
BASE_URL   = "https://raw.githubusercontent.com/DavisAdrian/CIS3120-Spring2026/main/"
BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

# ── Local / Colab destination paths ──────────────────────────────────────────
BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH   = "/content/Loan.csv"
DB_PATH     = "/content/library.db"

print("Libraries imported.")
print(f"  Book   -> {BOOK_URL}")
print(f"  Member -> {MEMBER_URL}")
print(f"  Loan   -> {LOAN_URL}")
print(f"  DB     -> {DB_PATH}")

Libraries imported.
  Book   → https://raw.githubusercontent.com/DavisAdrian/CIS3120-Spring2026/main/Book.csv
  Member → https://raw.githubusercontent.com/DavisAdrian/CIS3120-Spring2026/main/Member.csv
  Loan   → https://raw.githubusercontent.com/DavisAdrian/CIS3120-Spring2026/main/Loan.csv
  DB     → /content/library.db


## Download CSV Files

Downloads `Book.csv`, `Member.csv`, and `Loan.csv` directly from GitHub
so the notebook runs without any manual file uploads.

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


## Create Database and Tables

In [3]:
# Remove any leftover database from a previous run so we start fresh
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

# ── Book ───────────
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT NOT NULL,
    title   TEXT NOT NULL,
    author  TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

# ── Member ─────────
conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id        INTEGER NOT NULL,
    firstname TEXT    NOT NULL,
    lastName  TEXT    NOT NULL,
    PRIMARY KEY (id)
);
""")

# ── Loan ──────────
conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo       TEXT    NOT NULL,
    id           INTEGER NOT NULL,
    dateBorrowed TEXT    NOT NULL,
    dateReturned TEXT,
    dateDue      TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created successfully.")

# Verify all three tables exist
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()
print("Tables in database:", [t[0] for t in tables])


Tables created successfully.
Tables in database: ['Book', 'Loan', 'Member']


## Load Book Data

In [4]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0]
print(f'Book rows loaded: {count}  (expected 11)')


Book rows loaded: 11  (expected 11)


## Load Member Data

In [5]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0]
print(f'Member rows loaded: {count}  (expected 4)')


Member rows loaded: 4  (expected 4)


## Load Loan Data

**Data quality handling:** `dateReturned` is empty for unreturned books.
Empty strings are converted to `None` so SQLite stores them as `NULL`.

In [ ]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',
            (row['callNo'], int(row['id']),
             row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0]
print(f'Loan rows loaded: {count}  (expected 4)')


Loan rows loaded: 4  (expected 4)


## Query 1 — All Books

Retrieve all columns from the `Book` table, ordered alphabetically by author name.

In [7]:
print("Query 1: All books ordered by author name\n")
rows = conn.execute("""
    SELECT callNo, title, author
    FROM   Book
    ORDER  BY author;
""").fetchall()

print(f"{'Call No':<30} {'Title':<55} {'Author'}")
print("-" * 100)
for callNo, title, author in rows:
    print(f"{callNo:<30} {title:<55} {author}")

print(f"\nTotal rows returned: {len(rows)}")


Query 1: All books ordered by author name

Call No                        Title                                                   Author
----------------------------------------------------------------------------------------------------
R 487 T35 1967                 Medicine in medieval England.                           Charles H Talbot
QA 76.9 D26H39 1996            Data model patterns : conventions of thought            David Hay
CB 351 M293 1983               Atlas of medieval Europe                                Donald Matthew
HQ 1143 P68 1975               Medieval women                                          Eileen Power
PC 14 V48 1965                 Medieval miscellany                                     Frederick Whitehead
QA 76.73 S67C435 2004          Joe Celko's Trees and hierarchies in SQL for smarties   Joe Celko
QA 76.73 S67C46 1997           Joe Celko's SQL puzzles & answers                       Joe Celko
QA 76.9 D35C45 1999            Joe Celko's data & database

## Query 2 — Books Not Yet Returned

Retrieve the book title and borrowing member's full name for all loans where `dateReturned IS NULL` (book is still checked out). Requires a JOIN across all three tables.

In [8]:
print("Query 2: Books currently checked out (dateReturned IS NULL)\n")
rows = conn.execute("""
    SELECT B.title,
           M.firstname,
           M.lastName
    FROM   Loan   L
    JOIN   Book   B ON L.callNo = B.callNo
    JOIN   Member M ON L.id     = M.id
    WHERE  L.dateReturned IS NULL;
""").fetchall()

print(f"{'Title':<55} {'First Name':<15} {'Last Name'}")
print("-" * 85)
for title, first, last in rows:
    print(f"{title:<55} {first:<15} {last}")

print(f"\nTotal rows returned: {len(rows)}")


Query 2: Books currently checked out (dateReturned IS NULL)

Title                                                   First Name      Last Name
-------------------------------------------------------------------------------------
Joe Celko's SQL puzzles & answers                       David           Martin
Medieval medicine and the plague                        David           Martin

Total rows returned: 2


## Query 3 — Loan History for a Specific Book

Retrieve the full loan history for `callNo = 'R 141 E45 2006'`, showing the member's full name, `dateBorrowed`, `dateDue`, and `dateReturned`, ordered by `dateBorrowed` ascending.

In [9]:
print("Query 3: Loan history for 'R 141 E45 2006'\n")
rows = conn.execute("""
    SELECT M.firstname || ' ' || M.lastName AS member,
           L.dateBorrowed,
           L.dateDue,
           COALESCE(L.dateReturned, 'Not returned') AS dateReturned
    FROM   Loan   L
    JOIN   Member M ON L.id = M.id
    WHERE  L.callNo = 'R 141 E45 2006'
    ORDER  BY L.dateBorrowed ASC;
""").fetchall()

print(f"{'Member':<20} {'Borrowed':<22} {'Due':<22} {'Returned'}")
print("-" * 85)
for member, borrowed, due, returned in rows:
    print(f"{member:<20} {borrowed:<22} {due:<22} {returned}")

print(f"\nTotal rows returned: {len(rows)}")


Query 3: Loan history for 'R 141 E45 2006'

Member               Borrowed               Due                    Returned
-------------------------------------------------------------------------------------
Betty Freeman        4/1/2014 0:00          4/15/2014 0:00         4/15/2014 0:00
David Martin         4/30/2014 0:00         5/14/2014 0:00         Not returned

Total rows returned: 2


## Query 4 — Members Who Have Never Borrowed a Book

Retrieve members with **no record at all** in the `Loan` table (not even a completed loan). Uses a `LEFT JOIN` — rows where the right side is `NULL` indicate members absent from `Loan`.

In [10]:
print("Query 4: Members with no loan record\n")
rows = conn.execute("""
    SELECT M.id,
           M.firstname,
           M.lastName
    FROM   Member M
    LEFT JOIN Loan L ON M.id = L.id
    WHERE  L.id IS NULL;
""").fetchall()

print(f"{'ID':<5} {'First Name':<15} {'Last Name'}")
print("-" * 35)
for mid, first, last in rows:
    print(f"{mid:<5} {first:<15} {last}")

print(f"\nTotal rows returned: {len(rows)}")


Query 4: Members with no loan record

ID    First Name      Last Name
-----------------------------------
4     John            Martin

Total rows returned: 1


## Query 5 — Count of Loans per Member

Retrieve each member's full name and their total loan count (including zero). Uses a `LEFT JOIN` from `Member` to `Loan` with `COUNT()` and `GROUP BY`, ordered by loan count descending.

In [11]:
print("Query 5: Total loans per member (including members with 0 loans)\n")
rows = conn.execute("""
    SELECT M.firstname || ' ' || M.lastName AS member,
           COUNT(L.callNo) AS total_loans
    FROM   Member M
    LEFT JOIN Loan L ON M.id = L.id
    GROUP  BY M.id, M.firstname, M.lastName
    ORDER  BY total_loans DESC;
""").fetchall()

print(f"{'Member':<25} {'Total Loans'}")
print("-" * 40)
for member, total in rows:
    print(f"{member:<25} {total}")

print(f"\nTotal rows returned: {len(rows)}")


Query 5: Total loans per member (including members with 0 loans)

Member                    Total Loans
----------------------------------------
David Martin              2
John Smith                1
Betty Freeman             1
John Martin               0

Total rows returned: 4


## Query 6 — Free-Choice Analytical Query

**Business question:** Which books have been borrowed more than once, and how many times has each been checked out?

**Why it is useful:** A library manager can use this to identify the most popular titles and decide whether to acquire additional copies, prioritise holds, or feature them in displays. Titles with multiple loans signal high member demand relative to the rest of the collection.


In [ ]:
print("Query 6: Books borrowed more than once (demand analysis)\n")
rows = conn.execute("""
    SELECT B.title,
           B.author,
           COUNT(*) AS times_borrowed
    FROM   Loan L
    JOIN   Book B ON L.callNo = B.callNo
    GROUP  BY L.callNo
    HAVING COUNT(*) > 1
    ORDER  BY times_borrowed DESC;
""").fetchall()

print(f"{'Title':<50} {'Author':<20} {'Times Borrowed'}")
print("-" * 90)
for title, author, count in rows:
    print(f"{title:<50} {author:<20} {count}")

print(f"\nTotal rows returned: {len(rows)}")


## Summary

**Data quality observation:**  
The `dateReturned` column in `Loan.csv` uses an empty string to represent "not yet returned" rather than an explicit `NULL`. Without explicit handling in the loading code (converting `''` to `None`), SQLite would store an empty string instead of `NULL`, causing `IS NULL` filters in Queries 2 and 3 to silently return zero rows — a subtle bug that would be hard to diagnose later.

**Limitation of this dataset for a real library system:**  
The `Loan` table uses `(callNo, id, dateBorrowed)` as a composite primary key, which assumes a member cannot borrow the same book twice on the same date. In practice, a library system would also need a `copyNumber` column to distinguish multiple physical copies of the same title, and an `loanID` surrogate key to handle edge cases and simplify foreign-key references from reservation or fine tables.

---
*Notebook completed by Adrian Davis - CIS 3120 Spring 2026*


## Close Database Connection

In [ ]:
conn.close()
print("Database connection closed.")
